<a href="https://colab.research.google.com/github/bhuvaneshwaranyai2024-alt/Gokuchi_detection/blob/main/Email_Spam_Detection_Fresh_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Install libraries

In [1]:
%pip -q install scikit-learn pandas

## 2. Download and prepare a real public dataset

In [2]:
import os, re, tarfile, urllib.request, tempfile, random
from email import policy
from email.parser import BytesParser
import pandas as pd

random.seed(42)
CORPUS_URLS = {
    'spam': [
        'https://spamassassin.apache.org/old/publiccorpus/20030228_spam_2.tar.bz2',
        'https://spamassassin.apache.org/old/publiccorpus/20021010_spam.tar.bz2',
    ],
    'ham': [
        'https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham_2.tar.bz2',
        'https://spamassassin.apache.org/old/publiccorpus/20030228_hard_ham.tar.bz2',
        'https://spamassassin.apache.org/old/publiccorpus/20021010_easy_ham.tar.bz2',
    ],
}

def parse_email(raw_bytes):
    try:
        msg = BytesParser(policy=policy.default).parsebytes(raw_bytes)
        subject = str(msg.get('subject', '') or '')
        parts = []
        if msg.is_multipart():
            for part in msg.walk():
                if part.get_content_type() == 'text/plain' and not part.get_content_disposition():
                    try: parts.append(str(part.get_content()))
                    except Exception: pass
        elif msg.get_content_type() == 'text/plain':
            try: parts.append(str(msg.get_content()))
            except Exception: pass
        body = '\\n'.join(parts)
        text = (f'Subject: {subject}\\n\\n{body}').strip()
        return re.sub(r'\\s+', ' ', text)[:12000].strip()
    except Exception:
        return ''

def download_archive(url, label):
    rows = []
    with tempfile.NamedTemporaryFile(suffix='.tar.bz2') as tmp:
        urllib.request.urlretrieve(url, tmp.name)
        with tarfile.open(tmp.name, 'r:bz2') as tar:
            for member in tar.getmembers():
                if not member.isfile() or member.size > 2_000_000: continue
                try:
                    f = tar.extractfile(member)
                    if f:
                        message = parse_email(f.read())
                        if len(message) >= 30:
                            rows.append({'text': message, 'label': label})
                except Exception:
                    pass
    return rows

rows, errors = [], []
for label, urls in CORPUS_URLS.items():
    for url in urls:
        try:
            batch = download_archive(url, label)
            print(label, url.split('/')[-1], '->', len(batch), 'emails')
            rows.extend(batch)
        except Exception as exc:
            errors.append(str(exc))
            print('Download failed:', url.split('/')[-1], str(exc)[:120])

if not rows:
    raise RuntimeError('Could not download the public corpus. Check internet access and rerun this cell.')

data = pd.DataFrame(rows)
data['normalized'] = data['text'].str.lower().str.replace(r'\\W+', ' ', regex=True).str.strip()
data = data.drop_duplicates('normalized').drop(columns='normalized')
spam = data[data.label == 'spam']
ham = data[data.label == 'ham']
n = min(500, len(spam), len(ham))
if n < 100:
    raise RuntimeError(f'Not enough examples from both labels: spam={len(spam)}, ham={len(ham)}')
df = pd.concat([spam.sample(n=n, random_state=42), ham.sample(n=n, random_state=42)])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.to_csv('email_spam_dataset.csv', index=False)
print('Dataset saved:', df.shape)
print(df['label'].value_counts())
display(df.head())

spam 20030228_spam_2.tar.bz2 -> 1345 emails
spam 20021010_spam.tar.bz2 -> 483 emails
ham 20030228_easy_ham_2.tar.bz2 -> 1400 emails
ham 20030228_hard_ham.tar.bz2 -> 249 emails
ham 20021010_easy_ham.tar.bz2 -> 2550 emails
Dataset saved: (1000, 2)
label
ham     500
spam    500
Name: count, dtype: int64


,text,label
0,Subject: inbox mail notification broken\n\n\nH...,ham
1,Subject: RE: [ILUG] redhat kickstart\n\nCheck ...,ham
2,Subject: Re: Tech's Major Decline\n\n\nI'm not...,ham
3,Subject: 26-hour days...\n\nURL: http://jeremy...,ham
4,Subject: Dirty Knickers n Lollipops\n\n,spam


## 3. Train and evaluate the model

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.20, random_state=42, stratify=df['label']
)
features = FeatureUnion([
    ('word', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=60000, sublinear_tf=True)),
    ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=2, max_features=40000, sublinear_tf=True)),
])
model = Pipeline([('features', features), ('classifier', LinearSVC(class_weight='balanced', random_state=42))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, pred):.4f} ({accuracy_score(y_test, pred)*100:.2f}%)')
print(classification_report(y_test, pred, labels=['ham', 'spam'], zero_division=0))
print('Confusion matrix (actual rows: ham, spam; predicted columns: ham, spam):')
print(confusion_matrix(y_test, pred, labels=['ham', 'spam']))

Test accuracy: 0.9750 (97.50%)
              precision    recall  f1-score   support

         ham       1.00      0.95      0.97       100
        spam       0.95      1.00      0.98       100

    accuracy                           0.97       200
   macro avg       0.98      0.97      0.97       200
weighted avg       0.98      0.97      0.97       200

Confusion matrix (actual rows: ham, spam; predicted columns: ham, spam):
[[ 95   5]
 [  0 100]]


## 4. Download the dataset for GitHub

In [4]:
from google.colab import files
files.download('email_spam_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. Next steps

Upload `email_spam_dataset.csv` to the root of your GitHub repository, beside `app.py` and `requirements.txt`. You do **not** need to upload `email_spam_model.joblib` with this version. The Streamlit app trains its model from the CSV.